In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/b_Conflict_Features/Conflict_Topics/BERT', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [4]:
import pandas as pd
from transformers import pipeline
from transformers import RobertaModel, RobertaTokenizer, RobertaForSequenceClassification
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [3]:
# Load the ACLED notes
notes = pd.read_csv(os.path.join(project_root_dir, '1_Features/b_Conflict_Features/Conflict_Topics/BERT/notes.csv'))

In [4]:
# Define a custom to function to run the classifier

def get_labels(notes_with_ids, model, candidate_labels):
    results = []

    classifier = pipeline("zero-shot-classification", model=model)

    for index, row in notes_with_ids.iterrows():
        event_id = row['event_id']
        note = row['note']  
        sequence_to_classify = note
        output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
        scores_for_note = {label: score for label, score in zip(output["labels"], output["scores"])}
        scores_for_note['Event ID'] = event_id
        results.append(scores_for_note)

    df = pd.DataFrame(results)
    df.set_index('Event ID', inplace=True)
    return df




In [8]:
# Define the model
model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Define the classifier pipeline
classifier = pipeline("zero-shot-classification", model = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

# Create the topics to classify on
candidate_labels = ["environment", "unions", "gun-violence", "healthcare"]

In [9]:
# Run the classifier function on the ACLED notes, the specified model, and the defined topic labels
topics = get_labels(notes, model, candidate_labels)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [10]:
topics = topics.apply(lambda row: row == row.max(), axis=1).astype(int)

In [11]:
# Add categotical label variable to the topics dataframe
topics['label'] = topics.idxmax(axis=1)

# Add additional numerical label variable to the topics dataframe
topics['label_num'] = topics['label'].map({'environment': 1, 'unions': 2, 'gun-violence': 3, 'healthcare': 4, 'racial-justice': 5})


In [5]:
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "topics.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

topics.to_csv(output_path_1, index=False)

NameError: name 'topics' is not defined